# 06 — Exploratory Report Claims: Audit

MallWorld audit (2026-10)

---

## Scope

This notebook audits *Emergent Structure in Recurring Dream Environments* (archived notebook 04), an exploratory report with no framework predictions. It checks the claims that depend on a computation that can fail:
- X1: interaction × location associations (the archived join);
- X2: dream "archetypes" from k-means;
- X3: principal components;
- X4: location "self-persistence" in transitions;
- X5: association rules and entity–interaction links that are built into the schema.

The atmospheric trajectory (§3.4: first half −0.337, second half −0.404) agrees in direction with the corrected within-dream decline (notebook 02 C11; notebook 04 S4). The location–atmosphere table (§3.1) is audited in notebook 05 R1.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
L = mw.load_locations(); C = mw.load_connections(); I = mw.load_interactions(); E = mw.load_entities()
rng = np.random.default_rng(6)

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


## X1 — Interaction × location

**Old.**
- "Social interactions are dramatically elevated in malls (z = +22.82), while transactions are dramatically suppressed in malls (z = −18.16)."
- "Escape − mall (z = −14.94, n = 136,422)".

The archived code merged interactions with locations on `location_id` alone.

In [2]:
Ix, Lx = mw.load_interactions("extractable"), mw.load_locations("extractable")
wrong = Ix.merge(Lx[["location_id", "location_type"]], on="location_id", how="left")
print(f"Archived join: {len(Ix):,} interactions -> {len(wrong):,} rows")
def resid(df):
    ci = df["interaction_type"].value_counts(); cl = df["location_type"].value_counts()
    d = df[df["interaction_type"].isin(ci[ci >= 50].index) & df["location_type"].isin(cl[cl >= 100].index)]
    tab = pd.crosstab(d["interaction_type"], d["location_type"])
    chi, p, dof, exp = chi2_contingency(tab)
    return tab, (tab - exp) / np.sqrt(exp), chi, dof
for nm, df in [("archived join", wrong), ("correct key", Ix.merge(Lx[["post_id", "location_id", "location_type"]], on=["post_id", "location_id"], how="inner"))]:
    tab, z, chi, dof = resid(df)
    pick = {k: round(float(z.loc[k[0], k[1]]), 2) for k in [("social", "mall"), ("transaction", "mall"), ("transaction", "mall_store"), ("escape", "mall")] if k[0] in z.index and k[1] in z.columns}
    print(f"{nm:14s}: n = {int(tab.values.sum()):,}, chi2 = {chi:.1f}, df = {dof}; residuals {pick}")

Ip = I.merge(L[["post_id", "location_id", "location_type"]], on=["post_id", "location_id"], how="inner")
for it, lt in [("social", "mall"), ("transaction", "mall"), ("transaction", "mall_store"), ("escape", "mall")]:
    d = Ip.assign(y=(Ip["interaction_type"] == it).astype(float), x=(Ip["location_type"] == lt).astype(int))
    t = mw.odds_ratio_table(mw.clustered_logit(d, "y ~ x + log_words"), ["x"]).iloc[0]
    print(f"Primary, clustered: {it:11s} in {lt:10s} {100 * d.loc[d['x'] == 1, 'y'].mean():.1f}% vs elsewhere {100 * d.loc[d['x'] == 0, 'y'].mean():.1f}%; OR {t.OR:.2f} [{t.lo:.2f}-{t.hi:.2f}], p = {t.p:.2g}")

Archived join: 7,283 interactions -> 12,346,344 rows


archived join : n = 12,346,313, chi2 = 3850.5, df = 552; residuals {('social', 'mall'): 20.96, ('transaction', 'mall'): -23.55, ('transaction', 'mall_store'): 8.26, ('escape', 'mall'): -13.88}
correct key   : n = 4,737, chi2 = 739.7, df = 72; residuals {('social', 'mall'): 1.45, ('transaction', 'mall'): -2.46, ('transaction', 'mall_store'): 15.03, ('escape', 'mall'): 0.19}


Primary, clustered: social      in mall       14.3% vs elsewhere 12.9%; OR 1.14 [0.92-1.42], p = 0.22
Primary, clustered: transaction in mall       2.2% vs elsewhere 3.8%; OR 0.57 [0.34-0.96], p = 0.036
Primary, clustered: transaction in mall_store 18.3% vs elsewhere 2.5%; OR 8.84 [6.50-12.03], p = 7.6e-44
Primary, clustered: escape      in mall       12.4% vs elsewhere 12.4%; OR 1.02 [0.82-1.28], p = 0.85


### Findings — X1

**The archived residuals come from the join bug.**
- Merging on `location_id` alone turned 7,283 interactions into 12,346,344 rows.
- On that join the archived pattern reproduces: social × mall +20.96, transaction × mall −23.55, escape × mall −13.88.
- With the correct key (4,737 interactions at typed locations) the residuals are +1.45, −2.46 and +0.19.

**Corrected** (primary population, clustered, length-adjusted):

| Interaction | Location | Rate there | Rate elsewhere | OR (95% CI) |
|---|---|---|---|---|
| social | mall | 14.3% | 12.9% | 1.14 (0.92–1.42), p = 0.22 |
| transaction | mall | 2.2% | 3.8% | 0.57 (0.34–0.96), p = 0.036 |
| transaction | mall store | 18.3% | 2.5% | 8.84 (6.50–12.03) |
| escape | mall | 12.4% | 12.4% | 1.02, p = 0.85 |

**Withdrawn:**
- "Social interactions are dramatically elevated in malls (z = +22.82)";
- "escape suppressed in malls";
- §4.3 "The Mall as Social Space".

What remains is that transactions happen in stores, which needs no analysis.

## X2 — "Four dream archetypes" from k-means

**Old.** k = 4 chosen "based on the elbow". The archived inertia fell steadily from k = 2 to k = 9 (34,707 → 24,172) with no elbow, and no other criterion was computed. The features mix counts, which track narrative length, with shares that are set to 0 when a field is absent.

In [3]:
P = posts[posts["primary"]].set_index("post_id")
feat = pd.DataFrame(index=P.index)
feat["n_locations"] = L.groupby("post_id").size(); feat["n_connections"] = C.groupby("post_id").size()
feat["n_interactions"] = I.groupby("post_id").size(); feat["n_entities"] = E.groupby("post_id").size()
for a in ["threatening", "eerie", "neutral", "welcoming", "peaceful", "chaotic", "oppressive"]:
    feat[f"atmos_{a}"] = L.assign(x=L["atmosphere"] == a).groupby("post_id")["x"].mean()
for s in ["solid", "shifting"]:
    feat[f"stab_{s}"] = L.assign(x=L["reality_stability"] == s).groupby("post_id")["x"].mean()
for m in ["directed_active", "passive", "fleeing", "wandering"]:
    feat[f"mode_{m}"] = C.assign(x=C["transit_mode"] == m).groupby("post_id")["x"].mean()
feat["dir_h"] = C.assign(x=C["direction"] == "horizontal").groupby("post_id")["x"].mean()
feat["dir_up"] = C.assign(x=C["direction"].isin(["up", "diagonal_up"])).groupby("post_id")["x"].mean()
feat["dir_down"] = C.assign(x=C["direction"].isin(["down", "diagonal_down"])).groupby("post_id")["x"].mean()
feat = feat.fillna(0)
X = StandardScaler().fit_transform(feat.values)
print(f"Feature matrix: {X.shape[0]} dreams x {X.shape[1]} features")
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    print(f"  k={k}: inertia {km.inertia_:.0f}, silhouette {silhouette_score(X, km.labels_, sample_size=1500, random_state=0):.3f}")
base = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)
aris = []
for i in range(50):
    idx = rng.choice(len(X), len(X), replace=True)
    km = KMeans(n_clusters=4, random_state=i, n_init=10).fit(X[idx])
    aris.append(adjusted_rand_score(base.labels_, km.predict(X)))
print(f"k = 4: bootstrap stability (adjusted Rand vs full-sample solution): median {np.median(aris):.2f}, 5th pct {np.percentile(aris, 5):.2f}")
lab = pd.Series(base.labels_, index=feat.index)
print("Cluster sizes:", lab.value_counts().sort_index().to_dict())
print("Median words by cluster:", P["words"].groupby(lab).median().round(0).to_dict())
eta = P["log_words"].groupby(lab).transform("mean").var() / P["log_words"].var()
print(f"Share of log word count variance between clusters (eta2): {eta:.3f}")

Feature matrix: 1918 dreams x 20 features
  k=2: inertia 34567, silhouette 0.119


  k=3: inertia 32423, silhouette 0.110
  k=4: inertia 30673, silhouette 0.118


  k=5: inertia 29114, silhouette 0.133
  k=6: inertia 27910, silhouette 0.137


  k=7: inertia 26473, silhouette 0.143


  k=8: inertia 25230, silhouette 0.157


k = 4: bootstrap stability (adjusted Rand vs full-sample solution): median 0.70, 5th pct 0.52
Cluster sizes: {0: 790, 1: 231, 2: 185, 3: 712}
Median words by cluster: {0: 163.0, 1: 521.0, 2: 164.0, 3: 98.0}
Share of log word count variance between clusters (eta2): 0.266


### Findings — X2

**No cluster structure.**
- The silhouette is 0.11–0.16 for every k from 2 to 8. Values below 0.25 indicate no substantial structure.
- k = 4 is not preferred by any criterion: silhouette 0.118, against 0.157 at k = 8.
- Inertia falls smoothly, as in the archived run.
- The four-cluster partition is moderately reproducible under resampling (median adjusted Rand 0.70, 5th percentile 0.52). It is driven partly by length: median words are 521 in one cluster and 98 in another, and clusters account for 26.6% of the variance of log word count.

**Withdrawn:** "four distinct dream archetypes". They are cuts through a continuum, one of which separates long reports from short ones.

## X3 — Principal components

**Old.** "Four interpretable latent dimensions emerge." PC1 explained 15.3%. The number retained was not estimated (CLAUDE.md §4.10). Parallel analysis compares each eigenvalue with the 95th percentile of eigenvalues from random data of the same shape.

In [4]:
ev = np.sort(np.linalg.eigvalsh(np.corrcoef(X, rowvar=False)))[::-1]
rand = np.array([np.sort(np.linalg.eigvalsh(np.corrcoef(rng.standard_normal(X.shape), rowvar=False)))[::-1] for _ in range(200)])
thr = np.percentile(rand, 95, axis=0)
print("Eigenvalues  :", np.round(ev[:8], 2))
print("Random 95th  :", np.round(thr[:8], 2))
print(f"Components retained by parallel analysis: {int(np.argmax(ev <= thr))}; by Kaiser (eigenvalue > 1): {int((ev > 1).sum())}; PC1 explains {100 * ev[0] / ev.sum():.1f}%")

Eigenvalues  : [3.06 1.71 1.46 1.35 1.18 1.15 1.08 1.03]
Random 95th  : [1.21 1.17 1.15 1.12 1.1  1.08 1.07 1.05]
Components retained by parallel analysis: 7; by Kaiser (eigenvalue > 1): 8; PC1 explains 15.3%


### Findings — X3

- **Dimensionality.** Parallel analysis retains 7 components and the Kaiser criterion 8. PC1 explains 15.3%, the same as the archived run.
- **No low-dimensional structure.** The 20 dream features do not reduce to a few latent dimensions.
- **Withdrawn:** "four interpretable latent dimensions". The number four was imposed, not estimated.

## X4 — "Schools lead to more school locations 6.38× more than random"

In [5]:
O = L.sort_values(["post_id", "list_index"])
seq = O.groupby("post_id")["location_type"].apply(list)
def lift(seqs, t):
    pairs = [(a, b) for q in seqs for a, b in zip(q[:-1], q[1:])]
    base = np.mean([b == t for _, b in pairs]); cond = [b == t for a, b in pairs if a == t]
    return np.mean(cond) / base, len(cond)
for t in ["school", "restaurant", "mall_store", "mall", "house"]:
    obs, n_ = lift(seq, t)
    shuf = [lift(seq.map(lambda q: list(rng.permutation(q))), t)[0] for _ in range(100)]
    print(f"{t:11s}: lift {obs:.2f} (n = {n_} transitions from it); with each dream's order shuffled {np.mean(shuf):.2f}")

school     : lift 3.79 (n = 228 transitions from it); with each dream's order shuffled 2.32


restaurant : lift 5.49 (n = 146 transitions from it); with each dream's order shuffled 3.58


mall_store : lift 3.99 (n = 476 transitions from it); with each dream's order shuffled 2.71


mall       : lift 1.32 (n = 719 transitions from it); with each dream's order shuffled 0.85


house      : lift 3.36 (n = 269 transitions from it); with each dream's order shuffled 1.95


### Findings — X4

- **Archived lifts.** Restaurant reproduces (5.49 vs 5.50); school does not (3.79 vs 6.38).
- **Most of the "self-persistence" is composition.** Dreams that contain a school contain several school locations. Shuffling the order of locations within each dream keeps lifts of 2.32 (school) and 3.58 (restaurant).
- **What adjacency adds.** Consecutive same-type locations are about 1.5 times as common as the dream's composition alone produces.
- **Descriptive.** Same-type neighbours come partly from how the coder splits one setting into several locations. For example, a school with a hallway and a classroom becomes two `school` locations.

## X5 — Rules and links that the schema builds in

In [6]:
mw.check_values("difficulty", ["blocked"]); mw.check_values("traversal_outcome", ["failed"]); mw.check_values("transit_mode", ["instant"])
b = C[C["difficulty"] == "blocked"]
print(f"blocked -> traversal failed: {100 * (b['traversal_outcome'] == 'failed').mean():.1f}% of {len(b)} blocked connections")
tp = C[C["connection_type"] == "teleport"]
print(f"teleport -> transit instant: {100 * (tp['transit_mode'] == 'instant').mean():.1f}% of {len(tp)} teleports")
EI = E.drop_duplicates(["post_id", "interaction_index", "entity_type"])
th = EI[EI["entity_type"] == "threat"]
print(f"interactions involving a threat that are escape or conflict: {100 * th['interaction_type'].isin(['escape', 'conflict']).mean():.1f}% of {len(th)}; "
      f"all interactions: {100 * I['interaction_type'].isin(['escape', 'conflict']).mean():.1f}%")

blocked -> traversal failed: 86.0% of 43 blocked connections
teleport -> transit instant: 95.0% of 60 teleports
interactions involving a threat that are escape or conflict: 74.9% of 299; all interactions: 18.0%


### Findings — X5

These associations are produced by the categories themselves:
- "Blocked difficulty → failed outcome": 86.0% of 43 blocked connections.
- "Teleportation is instant": 95.0% of 60 teleports.
- "Threats → escape or conflict": 74.9% of 299 threat interactions, against 18.0% of all interactions.

A blocked passage that is traversed successfully, or a non-instant teleport, would be a coding contradiction. A threat is a hostile entity. These rules show that the coder is consistent. They are not findings about dream structure (CLAUDE.md §4.5).